# Interpreting the selected job-posting SVM

Which words and phrases push a posting's score toward **Fake** or **Real**? We fit the Step 2 configuration retained in Step 3 on the development postings. We inspect the raw SVM because these coefficients determine its score before Platt calibration.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from fakejob.data import load_split
from fakejob.features import STEP2_CONFIG, build_pipeline, posting_features

# Reconstruct the project's fixed split using the same scikit-learn version.
if sklearn.__version__ != "1.6.1":
    raise RuntimeError("Use scikit-learn 1.6.1 to reproduce the project split.")
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="IProgress not found.*")
    split = load_split()
dev = split.model_df.loc[split.dev_index]
assert len(dev) == 15001 and dev.index.intersection(split.test_index).empty
X = posting_features(dev)
y = dev["fraudulent"]
print(f"Development postings: {len(dev):,} ({y.sum():,} Fake, {(y == 0).sum():,} Real)")

Development postings: 15,001 (724 Fake, 14,277 Real)


In [2]:
# Fit all feature blocks together so the weights belong to the selected model.
model = build_pipeline(STEP2_CONFIG).fit(X, y)
transformer, svm = model.steps[0][1], model.steps[-1][1]
assert list(svm.classes_) == [0, 1]
word = transformer.named_transformers_["word"]
terms = word.get_feature_names_out()
weights = svm.coef_[0, transformer.output_indices_["word"]]
assert len(weights) == len(terms)
print(f"Word and phrase coefficients available: {len(terms):,}")

Word and phrase coefficients available: 273,800


## Words and phrases with the strongest weights

Each table lists the five highest and five lowest coefficients within its feature type. A positive coefficient increases the Fake score when the term is present. The contribution for one posting equals the coefficient multiplied by that posting's TF-IDF value. The decision also uses character fragments and structured fields. Counts show how many development postings of each class contain the term.

In [3]:
# Count each matching posting once per term using the fitted word vocabulary.
matrix = word.transform(X["combined_text"]).tocsc()
signals = pd.DataFrame({
    "Term": terms, "Weight": weights,
    "Fake postings": matrix[y.to_numpy() == 1].getnnz(axis=0),
    "Real postings": matrix[y.to_numpy() == 0].getnnz(axis=0),
})
for title, subset in signals.groupby(np.where(signals["Term"].str.contains(" "), "Two-word phrases", "Single words")):
    print(title)
    extremes = pd.concat([subset.nlargest(5, "Weight").assign(Direction="Fake"),
                          subset.nsmallest(5, "Weight").assign(Direction="Real")])
    display(extremes.set_index(["Direction", "Term"]).round(3))

Single words


Weight  Fake postings  Real postings
Direction Term                                                
Fake      northwestern     0.761              1              3
          rohan            0.667              3              0
          link             0.630             53            191
          administrative   0.578             94            453
          assistant        0.559             81            598
Real      it              -0.481            111           6415
          web             -0.432             24           2677
          our             -0.412            387          11130
          of              -0.347            567          13547
          companies       -0.340             85           3627

Two-word phrases


Weight  Fake postings  Real postings
Direction Term                                                      
Fake      using below            0.794             34              0
          below link             0.788             35              1
          apply using            0.746             30              2
          junior project         0.744              1              5
          executive sales        0.741              1              5
Real      will be               -0.398            135           5055
          is looking            -0.371             21           1304
          your resume           -0.339             29            452
          company in            -0.334              5            387
          sales representative  -0.318              2            305

## Reading the signals in context

Inspect application instructions, a rare name-like term, and a phrase associated with Real. For comparison, also look up two payment-related phrases suggested before this analysis. Group counts indicate whether matches are concentrated in similar descriptions. Excerpts use the first matching development posting in each class and show normalized tokens in their original order.

In [4]:
tokenize = word.build_tokenizer()
preprocess = word.build_preprocessor()
checks = ["below link", "rohan", "is looking", "registration fee", "wire transfer"]
groups = split.groups.loc[dev.index]
for term in checks:
    column = word.vocabulary_.get(term)
    if column is None:
        print(f"{term!r}: outside the fitted word vocabulary.")
        continue
    positions = matrix[:, column].nonzero()[0]
    print(f"{term!r} | weight {weights[column]:+.3f} | {len(positions)} postings | "
          f"{groups.iloc[positions].nunique()} description groups")
    phrase = term.split()
    for label, name in [(1, "Fake"), (0, "Real")]:
        matches = positions[y.iloc[positions].to_numpy() == label]
        if len(matches):
            pos = matches[0]
            tokens = tokenize(preprocess(X["combined_text"].iloc[pos]))
            start = next(i for i in range(len(tokens)) if tokens[i:i + len(phrase)] == phrase)
            excerpt = " ".join(tokens[max(0, start - 8):start + len(phrase) + 8])
            print(f"  {name} | job {dev['job_id'].iloc[pos]} | … {excerpt} …")

'below link' | weight +0.788 | 36 postings | 25 description groups
  Fake | job 585 | … quality improvement manager apply using below link url_8b28dea5804b323e29db29adaf9fa38b688e5d9db1f6cf5b6b2e924236a07272 the quality improvement manager is responsible for …
  Real | job 4811 | … the international graduates programme please click on the below link url_041db52b909170920e2c98ffe13587eb084c3d337b16a4f02f3ca33b6b398941 …
'rohan' | weight +0.667 | 3 postings | 3 description groups
  Fake | job 11038 | … migration center work with rohan to develop …
'is looking' | weight -0.371 | 1325 postings | 1086 description groups
  Fake | job 741 | … person that is team oriented and someone who is looking to share their knowledge while seeking new opportunities …
  Real | job 2 | … and excellent customer service our rapidly expanding business is looking for talented project manager to manage the successful …
'registration fee': outside the fitted word vocabulary.
'wire transfer' | weight -0.005 | 39 

## What the evidence supports

- **Application instructions.** “Using below”, “below link” and “apply using” can overlap in a single instruction. Their positive weights point to wording worth reviewing. A repeated template can produce all three signals.
- **Possible dataset shortcuts.** “Rohan” appears in only three development postings. A name-like token with a high weight is a candidate for a masking experiment. The coefficient alone cannot establish that the model memorizes employers.
- **Language associated with Real.** “Is looking” occurs in both classes. Its negative coefficient describes how it changes this model's score alongside its other inputs. It cannot establish that an individual posting is genuine.
- **Follow-up experiment.** Compare the selected SVM with and without company profiles on the same grouped development folds. This tests how much the model depends on that field. A separate name-masking comparison can isolate the effect of names more directly.

## How much does the SVM rely on company profiles?

We compare the published Step 2 SVM scores with a new run that blanks the company profile before building features. Titles, job descriptions, company-logo flags and the other structured inputs are still available. Both runs use the same development rows, description groups, seed and nested threshold-selection procedure. The held-out test set is excluded. The comparison measures the contribution of the entire profile field, including names, writing style and company information.

In [1]:
from pathlib import Path
import sklearn
from fakejob.data import load_split
from fakejob.evaluation import repeated_group_cv
from fakejob.features import STEP2_CONFIG, build_pipeline, posting_features

# This experiment can start in a fresh kernel with the project's pinned version.
if sklearn.__version__ != "1.6.1":
    raise RuntimeError("Select a kernel with scikit-learn 1.6.1 to reproduce the project split.")
split = load_split()
dev = split.model_df.loc[split.dev_index]
y = dev["fraudulent"]
assert len(dev) == 15001 and dev.index.intersection(split.test_index).empty

# Remove the raw text before building both word and character features.
without_profile = posting_features(dev.assign(company_profile=""))
assert without_profile["company_profile"].eq("").all()
assert without_profile.index.equals(y.index)
print(f"Profiles removed from {dev['company_profile'].fillna('').str.strip().ne('').sum():,} of {len(dev):,} development postings.")

# Repeat the exact Step 2 evaluation with the ablated inputs.
ablated = repeated_group_cv(
    build_pipeline(STEP2_CONFIG), without_profile, y, split.groups.loc[y.index],
    name="Without company profile", n_splits=5, n_repeats=3, inner_splits=3,
    seed=0, n_jobs=2,
)

# Save the completed run so the comparison can be opened in a new session.
fold_file = Path("results") / "interpretation_without_profile_folds.csv"
ablated.folds.to_csv(fold_file, index=False)
print(f"Saved {len(ablated.folds)} folds to {fold_file}.")

/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Profiles removed from 12,203 of 15,001 development postings.


/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Saved 15 folds to results/interpretation_without_profile_folds.csv.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Reuse the original scores from feature_engineering.ipynb as the paired baseline.
keys = ["repeat", "fold"]
baseline_folds = pd.read_csv(Path("results") / "step2_combined_folds.csv").set_index(keys).sort_index()
# Read both runs from disk so this cell works on its own.
ablated_folds = pd.read_csv(Path("results") / "interpretation_without_profile_folds.csv").set_index(keys).sort_index()
pd.testing.assert_frame_equal(baseline_folds[["n_test", "n_fake"]], ablated_folds[["n_test", "n_fake"]])
assert len(ablated_folds) == 15 and ablated_folds.index.is_unique

metrics = ["average_precision", "precision", "recall", "f1"]
change = ablated_folds[metrics] - baseline_folds[metrics]
summary = pd.DataFrame({
    "With profile": baseline_folds[metrics].mean(),
    "Without profile": ablated_folds[metrics].mean(),
    "Change": change.mean(),
    "Folds lower without profile": (change < -1e-12).sum(),
})
display(summary.rename(index={"average_precision": "PR-AUC (AP)", "precision": "Fake precision",
                             "recall": "Fake recall", "f1": "Fake F1"}).round(4))

# Each repeat evaluates every development posting once across its five folds.
def missed_by_repeat(folds):
    detected = (folds["n_fake"] * folds["recall"]).round().astype(int)
    return (folds["n_fake"] - detected).groupby(level="repeat").sum()

missed = pd.DataFrame({"Missed with profile": missed_by_repeat(baseline_folds),
                       "Missed without profile": missed_by_repeat(ablated_folds)})
missed["Additional misses"] = missed["Missed without profile"] - missed["Missed with profile"]
display(missed)

,With profile,Without profile,Change,Folds lower without profile
PR-AUC (AP),0.8705,0.7228,-0.1477,15
Fake precision,0.8337,0.7338,-0.0999,15
Fake recall,0.7520,0.6005,-0.1515,15
Fake F1,0.7894,0.6557,-0.1337,15


,Missed with profile,Missed without profile,Additional misses
repeat,,,
0,179,278,99
1,168,281,113
2,177,293,116


### What changed when the profile was removed?

Fake recall fell from **75.2% to 60.1%**, and PR-AUC fell from **0.871 to 0.723**. All 15 paired folds showed lower recall and PR-AUC. In each repeat of 724 fake postings, removing profiles led to **99, 113 and 116 additional misses**, respectively. Each repeat evaluates the same development postings, so these counts should be read separately.

The company profile supplies useful information on this dataset. Removing it simultaneously removes employer names, template wording and the distinction between filled and empty profiles. To test whether names specifically explain part of the gain, retain the profile text and mask names in a separate paired experiment.